# 府文本嵌入

对 207 个府文本做滑动窗口 BERT 嵌入。

采用 512 token 窗口 / 128 步长的滑动，SentenceTransformer 嵌入后保存。

## 导入模块与配置

In [ ]:
import os
import time
import numpy as np
import torch
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
from pathlib import Path

INPUT_DIR = Path("../data/明一统志文本/明一统志文本_chgis")
OUTPUT_DIR = Path("../data/fu_embeddings")

MODEL_NAME = "Jihuai/bert-ancient-chinese"
BATCH_SIZE = 32
MAX_LENGTH = 512
STRIDE = 128

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Use {DEVICE}")

## 准备待嵌入的窗口切片

按照最大长度和滑动步长将每个府的文本切片，存储到 `all_batches`

In [ ]:
# 文件列表
text_files = []
for root, dirs, files in os.walk(INPUT_DIR):
    for fn in sorted(files):
        if fn.endswith(".txt"):
            text_files.append(Path(root) / fn)
print(f"Files to process: {len(text_files)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

all_batches = []
for fpath in text_files:
    text = fpath.read_text("utf-8").strip()
    if not text:
        continue
    
    # 预定存储位置，如果没有就创建，如果已经存在就跳过
    rel = fpath.relative_to(INPUT_DIR)
    out_path = OUTPUT_DIR / rel.with_suffix(".npy")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        continue
    
    # 划分窗口
    encoding = tokenizer(text, truncation=False, return_offsets_mapping=False, add_special_tokens=False)
    input_ids = encoding["input_ids"]
    n_tokens = len(input_ids)

    windows = []
    for start in range(0, max(1, n_tokens - MAX_LENGTH + STRIDE), STRIDE):
        end = min(start + MAX_LENGTH, n_tokens)
        chunk = input_ids[start:end]
        window_text = tokenizer.decode(chunk, skip_special_tokens=True)
        windows.append(window_text)
    if not windows:
        windows = [tokenizer.decode(input_ids[:MAX_LENGTH], skip_special_tokens=True)]

    # all_batches 储存的是元组 (存储位置, 窗口列表)
    all_batches.append((out_path, windows))

all_windows_flat = []  
batch_map = []  
for out_path, windows in all_batches:
    batch_map.append((out_path, len(all_windows_flat), len(windows)))
    # 记录每个文件对应的窗口在all_windows_flat中的起始位置和数量
    all_windows_flat.extend(windows)
    # 把所有窗口展平到一个列表中，便于后续处理

n_total = len(all_windows_flat)
n_batches = (n_total + BATCH_SIZE - 1) // BATCH_SIZE
print(f"Total windows: {n_total:,}, {n_batches} batches")

## 嵌入和保存

将上面切片转成向量

In [ ]:
# 加载模型
print(f"Loading model: {MODEL_NAME}...")
model = SentenceTransformer(
    MODEL_NAME, 
    device=DEVICE,                               
    model_kwargs={"attn_implementation": "eager", "torch_dtype": "float32"}
)
model.eval()

# Warmup
print('Warmup...')
warmup = ('東至杭州府富陽縣界二百里西至直隷徽州府歙縣界二百三十里南至金華府蘭溪縣界'
          '五十里北至杭州府於潛縣界一百二十里自府治至南京一千一百七十里至京師四千四百二十八里')
for _ in range(3):
    _ = model.encode(warmup)
if DEVICE == 'mps':
    torch.mps.synchronize()
    torch.mps.empty_cache()
print('Warmup done ✓')

# 计算所有窗口的嵌入
embeddings_all = []
for bi in range(n_batches):
    start_idx = bi * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, n_total)
    batch_windows = all_windows_flat[start_idx:end_idx]
    emb = model.encode(batch_windows)
    embeddings_all.append(emb)

    if DEVICE == "mps":
        torch.mps.synchronize()
        torch.mps.empty_cache()
        if (bi + 1) % 10 == 0:
            time.sleep(0.3)

    if (bi + 1) % 50 == 0 or (bi + 1) == n_batches:
        print(f"  [{bi + 1}/{n_batches}] {(bi + 1) / n_batches * 100:.0f}%")

all_embeddings = np.concatenate(embeddings_all, axis=0)

# 根据 batch_map 把每个府的嵌入保存到对应的文件
total_windows = 0
for out_path, start_idx, n_windows in batch_map:
    emb = all_embeddings[start_idx:start_idx + n_windows]
    np.save(out_path, emb)
    total_windows += n_windows

print(f"Saved {len(batch_map)} files, {total_windows:,} windows ")
